# VLM QLoRA Training — Google Colab Session

## Before running — Google Drive setup (one-time)

Organise your Drive like this:

```
MyDrive/
  vlm_training/
    projector/
      projector_stage1.pt          ← upload once
    session_state/
      lora_stepXXXX/               ← latest checkpoint folder
        adapter_config.json
        adapter_model.safetensors
        train_state.pt
      cls_head.pt
      faiss_index/                 ← copy from Kaggle session_state
      stage2.jsonl                 ← training log
```

**Colab Secrets** (key icon in left panel → Add new secret):
- `HF_TOKEN`
- `SEMANTIC_SCHOLAR_API_KEY`

**Session 1:** Set `IS_FIRST_SESSION = True` (no prior checkpoint on Drive).

**Session 2+:** Set `IS_FIRST_SESSION = False` (restores from Drive).


In [ ]:
# ── USER CONFIG — edit these before each session ──────────────────────────────

IS_FIRST_SESSION = False  # True = session 1 (no checkpoint yet)
                           # False = session 2+ (restores from Drive)

# Google Drive base path — matches the folder structure above
DRIVE_BASE = "/content/drive/MyDrive/vlm_training"

# Training config — full speed (4GB hypothesis already confirmed)
MAX_PAIRS  = 4000
EPOCHS     = 3
GRAD_ACCUM = 4      # full T4 speed
SAVE_EVERY = 250
LR         = 2e-4

print("Config loaded.")
print(f"  Session type : {'FIRST' if IS_FIRST_SESSION else 'RESUME'}")
print(f"  Drive base   : {DRIVE_BASE}")
print(f"  Grad accum   : {GRAD_ACCUM}")


In [ ]:
# ── Mount Google Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted.")


In [ ]:
# ── GPU check ─────────────────────────────────────────────────────────────────
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Runtime → Change runtime type → T4 GPU")
gpu = torch.cuda.get_device_properties(0)
print(f"GPU : {gpu.name}")
print(f"VRAM: {gpu.total_memory / 1e9:.1f} GB")


In [ ]:
# ── Install packages ──────────────────────────────────────────────────────────
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("peft>=0.19.1")
pip("bitsandbytes>=0.49.2")
pip("accelerate>=1.13.0")
pip("faiss-cpu==1.13.2")
pip("sentence-transformers")
pip("python-dotenv")

print("Packages ready.")


In [ ]:
# ── Secrets & environment variables ───────────────────────────────────────────
import os, torch
from google.colab import userdata

os.environ["HF_TOKEN"]                 = userdata.get("HF_TOKEN")
os.environ["SEMANTIC_SCHOLAR_API_KEY"] = userdata.get("SEMANTIC_SCHOLAR_API_KEY")
os.environ["CUBLAS_WORKSPACE_CONFIG"]  = ":4096:8"
os.environ["MEDDIAG_MAX_VRAM_GB"]      = "14"

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

print("Secrets loaded, HF login OK.")
print("Full 16 GB mode — no VRAM cap (4GB hypothesis already confirmed).")


In [ ]:
# ── Clone repo ────────────────────────────────────────────────────────────────
import subprocess, os

REPO_URL = "https://github.com/Sreenjoyee/visual-language-model-research-qlora-cot-rag.git"
REPO_DIR = "/content/vlm"

if not os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "clone", "--depth=1", REPO_URL, REPO_DIR])
    print(f"Repo cloned → {REPO_DIR}")
else:
    # Discard ALL local changes before pulling (cell 9 re-applies patches after)
    subprocess.call(["git", "-C", REPO_DIR, "reset", "--hard", "HEAD"])
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "--ff-only"])
    print(f"Repo updated → {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"Working dir: {os.getcwd()}")


In [ ]:
# ── Copy assets from Google Drive into working tree ───────────────────────────
import shutil, re
from pathlib import Path

drive = Path(DRIVE_BASE)
MODELS_DIR = Path(REPO_DIR) / "models"
MODELS_DIR.mkdir(exist_ok=True)

# Always copy projector
proj_src = drive / "projector" / "projector_stage1.pt"
shutil.copy2(proj_src, MODELS_DIR / "projector_stage1.pt")
print(f"Projector copied  ({proj_src.stat().st_size / 1e6:.0f} MB)")

if not IS_FIRST_SESSION:
    state_src = drive / "session_state"

    # Restore FAISS index
    faiss_src = state_src / "faiss_index"
    faiss_dst = Path(REPO_DIR) / "faiss_index"
    if faiss_src.exists():
        if faiss_dst.exists(): shutil.rmtree(faiss_dst)
        shutil.copytree(faiss_src, faiss_dst)
        print(f"FAISS index restored → {faiss_dst}")

    # Restore latest LoRA checkpoint(s)
    ckpt_pattern = re.compile(r"lora_step(\d+)$")
    for item in state_src.iterdir():
        if ckpt_pattern.match(item.name) and item.is_dir():
            dst = MODELS_DIR / item.name
            if dst.exists(): shutil.rmtree(dst)
            shutil.copytree(item, dst)
            print(f"Checkpoint restored  → {dst.name}")

    # Restore ClassificationHead
    cls_src = state_src / "cls_head.pt"
    if cls_src.exists():
        shutil.copy2(cls_src, MODELS_DIR / "cls_head.pt")
        print("ClassificationHead   restored")

    # Restore training log
    logs_dir = Path(REPO_DIR) / "logs"
    logs_dir.mkdir(exist_ok=True)
    log_src = state_src / "stage2.jsonl"
    if log_src.exists():
        shutil.copy2(log_src, logs_dir / "stage2.jsonl")
        print("Training log         restored")

print("\nAssets ready.")


In [ ]:
# ── Fix pipeline state ────────────────────────────────────────────────────────
from pathlib import Path

state_path = Path(REPO_DIR) / "logs" / ".pipeline_state"
state_path.parent.mkdir(exist_ok=True)

if IS_FIRST_SESSION:
    state_path.write_text("\nstep0\nstep0\nstep0\nstep0\nstep0\nstep0\nstep0\nstep2\n")
    print("Pipeline state: FAISS will be rebuilt (session 1)")
else:
    print("Pipeline state: using repo defaults (FAISS already done)")

lock = Path(REPO_DIR) / "logs" / ".pipeline.lock"
lock.unlink(missing_ok=True)
print("Lock cleared.")


In [ ]:
# ── Patch run_pipeline.sh ─────────────────────────────────────────────────────
import re
from pathlib import Path

pipeline_sh = Path(REPO_DIR) / "run_pipeline.sh"
text = pipeline_sh.read_text()

text = re.sub(r"(GRAD_ACCUM=)\d+",   f"GRAD_ACCUM={GRAD_ACCUM}",  text)
text = re.sub(r"(MAX_PAIRS_S2=)\d+", f"MAX_PAIRS_S2={MAX_PAIRS}", text)

pipeline_sh.write_text(text)
print(f"run_pipeline.sh patched: GRAD_ACCUM={GRAD_ACCUM}, MAX_PAIRS_S2={MAX_PAIRS}")


In [ ]:
# ── Run training ──────────────────────────────────────────────────────────────
import subprocess, os, shutil, re
from pathlib import Path

env = os.environ.copy()
cmd = ["bash", "run_pipeline.sh", "--resume"]

print("Starting pipeline — full 16 GB mode...")
print("=" * 60)

proc = subprocess.Popen(
    cmd,
    cwd=REPO_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in proc.stdout:
        print(line, end="", flush=True)
except KeyboardInterrupt:
    proc.terminate()
    print("\n[notebook] Interrupted — saving checkpoint...")

proc.wait()
print(f"\n[notebook] Process exited with code {proc.returncode}")

# ── Auto-save to Google Drive ─────────────────────────────────────────────────
print("\n[notebook] Saving session state to Drive...")
drive_state = Path(DRIVE_BASE) / "session_state"
drive_state.mkdir(parents=True, exist_ok=True)
MODELS = Path(REPO_DIR) / "models"
LOGS   = Path(REPO_DIR) / "logs"

ckpt_pattern = re.compile(r"lora_step(\d+)$")
ckpts = sorted([(int(m.group(1)), p) for p in MODELS.iterdir() if (m := ckpt_pattern.match(p.name))])
for _, ckpt_path in ckpts[-2:]:
    dst = drive_state / ckpt_path.name
    if dst.exists(): shutil.rmtree(dst)
    shutil.copytree(ckpt_path, dst)
    print(f"  Saved {ckpt_path.name} → Drive")

for fname in ["cls_head.pt"]:
    src = MODELS / fname
    if src.exists():
        shutil.copy2(src, drive_state / fname)
        print(f"  Saved {fname} → Drive")

faiss_src = Path(REPO_DIR) / "faiss_index"
if faiss_src.exists():
    faiss_dst = drive_state / "faiss_index"
    if faiss_dst.exists(): shutil.rmtree(faiss_dst)
    shutil.copytree(faiss_src, faiss_dst)
    print("  Saved faiss_index/ → Drive")

log = LOGS / "stage2.jsonl"
if log.exists():
    shutil.copy2(log, drive_state / "stage2.jsonl")
    print("  Saved stage2.jsonl → Drive")

total = sum(f.stat().st_size for f in drive_state.rglob("*") if f.is_file())
print(f"\nDone — {total / 1e6:.0f} MB saved to {drive_state}")


In [ ]:
# ── Manual save (run this any time to force a Drive sync) ─────────────────────
import shutil, re
from pathlib import Path

drive_state = Path(DRIVE_BASE) / "session_state"
drive_state.mkdir(parents=True, exist_ok=True)
MODELS = Path(REPO_DIR) / "models"
LOGS   = Path(REPO_DIR) / "logs"

ckpt_pattern = re.compile(r"lora_step(\d+)$")
ckpts = sorted([(int(m.group(1)), p) for p in MODELS.iterdir() if (m := ckpt_pattern.match(p.name))])
for _, ckpt_path in ckpts[-2:]:
    dst = drive_state / ckpt_path.name
    if dst.exists(): shutil.rmtree(dst)
    shutil.copytree(ckpt_path, dst)
    print(f"Saved {ckpt_path.name}")

for fname in ["cls_head.pt"]:
    src = MODELS / fname
    if src.exists():
        shutil.copy2(src, drive_state / fname)
        print(f"Saved {fname}")

faiss_src = Path(REPO_DIR) / "faiss_index"
if faiss_src.exists():
    faiss_dst = drive_state / "faiss_index"
    if faiss_dst.exists(): shutil.rmtree(faiss_dst)
    shutil.copytree(faiss_src, faiss_dst)
    print("Saved faiss_index/")

log = LOGS / "stage2.jsonl"
if log.exists():
    shutil.copy2(log, drive_state / "stage2.jsonl")
    print("Saved stage2.jsonl")

total = sum(f.stat().st_size for f in drive_state.rglob("*") if f.is_file())
print(f"\nTotal in Drive: {total / 1e6:.0f} MB")


## Between-session checklist

Cell 10 auto-saves to Drive when training ends. But if you interrupt manually:

1. Run **cell 11** (manual save) to push latest checkpoint to Drive.
2. Next session: set `IS_FIRST_SESSION = False`, run all cells.

## Drive folder structure

```
MyDrive/vlm_training/
  projector/projector_stage1.pt   ← never changes
  session_state/
    lora_stepXXXX/                ← updated each session (2 most recent)
    cls_head.pt
    faiss_index/                  ← built once, reused
    stage2.jsonl                  ← appended each session
```

## Speed reference

| GPU | Expected speed | Steps per 12h session |
|---|---|---|
| T4 (16 GB) | ~8–10 s/step | ~4,300–5,400 |
| V100 (16 GB) | ~4–6 s/step | ~7,200–10,800 |
| A100 (40 GB) | ~1–3 s/step | ~14,400–43,200 |

Total steps remaining depends on where you resume from.
